<a href="https://colab.research.google.com/github/Shahzal-Rehman/2/blob/main/fine_tune_nyra_qwen3_4b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nyra + Qwen3-4B: QLoRA fine-tune

This notebook trains the synthetic Nyra examples with a temporary GPU, then exports a Q4_K_M GGUF for local Ollama use. In Colab choose Runtime > Change runtime type > T4 GPU if available. GPU access is not guaranteed. No private chats or personal files are used.

This tunes style and interaction patterns. Agent actions come from the local tools in Nyra, not from fine-tuning alone.

In [ ]:
%pip -q install --upgrade unsloth datasets trl


In [ ]:
import torch
from google.colab import files
from pathlib import Path
from datasets import load_dataset

# Check for active GPU
if not torch.cuda.is_available():
    print('WARNING: No GPU found. This notebook requires a GPU to run training cells. Please switch your runtime to T4 GPU (Runtime > Change runtime type > T4 GPU).')
else:
    print('GPU Active:', torch.cuda.get_device_name(0))

# Accessing the existing file in the directory
data_file = '/content/nyra_training.jsonl'

try:
    dataset = load_dataset('json', data_files=data_file, split='train')
    print('Loaded synthetic examples:', len(dataset))
except Exception as e:
    print('Could not load dataset from local path, attempting upload...')
    uploaded = files.upload()
    data_file = next((name for name in uploaded if name.endswith('.jsonl')), None)
    if data_file:
        dataset = load_dataset('json', data_files=data_file, split='train')
        print('Loaded synthetic examples:', len(dataset))
    else:
        print('Dataset file not found. Please upload nyra_training.jsonl')

GPU Active: Tesla T4


Generating train split: 0 examples [00:00, ? examples/s]

Loaded synthetic examples: 24


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTConfig, SFTTrainer
MODEL_NAME = 'unsloth/Qwen3.5-4B'
MAX_SEQ_LENGTH = 1024
model, tokenizer = FastLanguageModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False, load_in_16bit=True, full_finetuning=False)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.2: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [6]:
model = FastLanguageModel.get_peft_model(model, r=8, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'], lora_alpha=16, lora_dropout=0, bias='none', use_gradient_checkpointing='unsloth', random_state=3407)
def format_row(row):
    return {'text': tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=False)}
train_data = dataset.map(format_row, remove_columns=dataset.column_names)
trainer_args = SFTConfig(output_dir='nyra-qwen3.5-4b-lora', per_device_train_batch_size=1, gradient_accumulation_steps=4, num_train_epochs=3, learning_rate=5e-5, warmup_ratio=0.05, logging_steps=1, save_strategy='epoch', optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear', seed=3407, report_to='none', dataset_num_proc=1, dataset_text_field='text', max_length=MAX_SEQ_LENGTH)
trainer = SFTTrainer(model=model, args=trainer_args, train_dataset=train_data, processing_class=tokenizer)
trainer.train()


Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"]:   0%|          | 0/24 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 3 | Total steps = 18
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 10,616,832 of 4,549,882,368 (0.23% trained)


Step,Training Loss
1,2.574064
2,2.558761
3,2.501199
4,2.284415
5,2.238788
6,2.007362
7,2.112775
8,1.829760
9,1.897340
10,1.695159


Unsloth: Restored added_tokens_decoder metadata in nyra-qwen3.5-4b-lora/checkpoint-6/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in nyra-qwen3.5-4b-lora/checkpoint-12/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in nyra-qwen3.5-4b-lora/checkpoint-18/tokenizer_config.json.


TrainOutput(global_step=18, training_loss=1.8736221260494657, metrics={'train_runtime': 294.3294, 'train_samples_per_second': 0.245, 'train_steps_per_second': 0.061, 'total_flos': 246726555254784.0, 'train_loss': 1.8736221260494657, 'epoch': 3.0})

In [ ]:
# Save the small adapter and export a quantized model for local Ollama.
model.save_pretrained('nyra-qwen3-4b-lora')
tokenizer.save_pretrained('nyra-qwen3-4b-lora')
model.save_pretrained_gguf('nyra-qwen3-4b-gguf', tokenizer, quantization_method='q4_k_m')
gguf_files = list(Path('nyra-qwen3-4b-gguf').rglob('*.gguf'))
if not gguf_files: raise RuntimeError('GGUF export did not produce a model file.')
target = Path('nyra-qwen3-4b.Q4_K_M.gguf')
gguf_files[0].replace(target)
print('Exported:', target, 'GB:', round(target.stat().st_size / 1e9, 2))
files.download(str(target))


## Install it locally

Put the downloaded GGUF in Nyra's outputs folder. Use `Modelfile-finetuned` and update the filename in its `FROM` line if needed, then run `ollama create nyra-qwen3:4b -f Modelfile-finetuned` from PowerShell in that folder. The local host will prefer this tuned model when installed; otherwise it uses the original Qwen3 4B.

The GPU notebook sees only the synthetic training examples and model weights. Existing local chats are not uploaded.